# ch10 p275 인터넷 검색을 활용해 답변하는 챗봇 만들기

In [1]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model = "gpt-4o-mini")
model.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

AIMessage(content='로제(Rosé)의 최근 신곡은 2021년 3월에 발표된 "ON THE GROUND"와 "GONE"입니다. 이 노래들은 그녀의 첫 솔로 앨범인 "R"에 수록되어 있습니다. 이후 새로운 곡이나 앨범 발표가 있을 경우, 공식 소식이나 음악 플랫폼을 통해 확인하시는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 19, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3v3m1iq77QJk0n1zShZX4KnmiXmF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d613a828-2070-4146-b8a9-261dcc481f3e-0', usage_metadata={'input_tokens': 19, 'output_tokens': 83, 'total_tokens': 102, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

print(docs)

c:\Users\pc04-06\anaconda3\envs\my_llm\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


snippet: Nov 22, 2024 · 22일 오후 로제의 새 싱글 'number one girl' 음원과 뮤직비디오가 공개됐다. 이는 'APT.'와 마찬가지로 오는 12월 6일 발매되는 로제의 첫 번째 정규 앨범 'rosie'에도 수록될 …, title: 로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표 - 싱글리스트, link: https://www.slist.kr/news/articleView.html?idxno=598065;
snippet: Nov 22, 2024 · 걸그룹 블랙핑크의 멤버 로제가 22일 새 싱글 ‘넘버 원 걸 (Number one girl)’을 발표했다. 넘버 원 걸은 로제가 내달 6일에 발매하는 첫 정규앨범 ‘로지 (rosie)’의 선공개 …, title: 로제, 신곡 ‘넘버 원 걸’ 공개… 브루노 마스가 프로듀싱, link: https://biz.chosun.com/entertainment/enter_general/2024/11/22/Q6AJ6YGRUJDPNIRGX5BKHHUBBY/;
snippet: Nov 22, 2024 · 그룹 블랙핑크 로제가 신곡 ‘넘버 원 걸 (number one girl)’을 발표했다. 더블랙레이블은 22일 공식 SNS를 통해 로제의 새 싱글 ‘넘버 원 걸’ 음원과 뮤직비디오 공개 …, title: “팬들을 생각하며 쓴 곡”...로제, 선공개 싱글 ‘넘버 원 걸 ..., link: https://www.mk.co.kr/news/musics/11175523;
snippet: Dec 6, 2024 · 솔로 뮤지션으로 활약 중인 블랙핑크 로제가 아파트 신드롬에 이어 앨범 ‘rosie’를 발매했습니다., title: 오래 기다리셨습니다. 로제의 솔로 앨범 ‘rosie’가 왔어요 ..., link: https://www.fastpapermag.com/2024/12/06/오래-기다리셨습니다-로제의-솔로-앨범-rosie가-왔/


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | model

In [4]:
from langchain.memory import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory() 
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("최근 로제가 발표한 신곡은 무엇인가요?") 

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

content="로제가 최근 발표한 신곡은 '넘버 원 걸 (Number one girl)'입니다. 이 곡은 2024년 12월 6일 발매될 로제의 첫 번째 정규앨범 'rosie'에 수록될 예정입니다." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 537, 'total_tokens': 594, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3vAeaQM1qZwc8P3E79J1SSO0JCcY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--d9318887-0948-40a9-8473-b7dfc73a6b7c-0' usage_metadata={'input_tokens': 537, 'output_tokens': 57, 'total_tokens': 594, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
# DuckDuckGo API wrapper를 사용하여 검색할 때 검색 매개변수를 설정하기 위한 클래스 import
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 한국 지역("kr-kr")을 기준, 최근 일주일("w") 내의 검색 결과를 가져오도록 초기화
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")


# 검색 기능을 위한 DuckDuckGoSearchResults 초기화
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,      # 앞에서 정의한 API wrapper를 사용
    source="news",            # 뉴스 소스에서만 검색하도록 지정
    results_separator=';\n'   # 결과 항목 사이에 구분자 사용 (세미콜론과 줄바꿈)
)

# "로제의 신곡 APT에 대한 반응"을 검색하고 결과를 docs에 저장
docs = search.invoke("최근 로제가 발표한 신곡은 무엇인가요?")

# 검색 결과 출력
print(docs)

c:\Users\pc04-06\anaconda3\envs\my_llm\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


snippet: 4 days ago · 그룹 블랙핑크 로제가 미국의 주요 대중음악 시상식인 ‘2025 MTV 비디오 뮤직 어워즈’ (VMA)에서 총 8개 부문 후보에 올랐습니다. 공개된 후보 명단에 따르면, 로제는 브루노 …, title: 로제 ‘아파트’ MTV어워즈 8개 부문 후보 “충격 받아서 무슨 ..., link: https://v.daum.net/v/psQHXXj4KY;
snippet: 5 days ago · 바로 우리의 자랑스러운 아티스트, 블랙핑크의 로제 님이 미국 최고 권위의 대중음악 시상식 중 하나인 'MTV 비디오 뮤직 어워즈 (VMA)'에서 무려 8개 부문 후보에 오르는 기염을 토했다는 소식입니다. …, title: 로제가 전한 소식, VMA 8관왕 대기록의 서막#20대 #연예인, link: /videos/riverview/relatedvideo?q=최근+로제가+발표한+신곡은+무엇인가요?&filters=ex1:"ez2"&ru=/search?q=%EC%B5%9C%EA%B7%BC+%EB%A1%9C%EC%A0%9C%EA%B0%80+%EB%B0%9C%ED%91%9C%ED%95%9C+%EC%8B%A0%EA%B3%A1%EC%9D%80+%EB%AC%B4%EC%97%87%EC%9D%B8%EA%B0%80%EC%9A%94%3F&filters=ex1%3A%22ez2%22&mmscn=vwrc&mid=324EDAEFC318920C86D0324EDAEFC318920C86D0&FORM=WRVORC&ntb=1&msockid=896acf2877ea11f0abcb483a7205f585;
snippet: 5 days ago · 세스 베일리 부차관보 대행, DPAA 행사서 밝혀 미국 국무부 당국자가 북한 비핵화가 아닌 주제에 대해서는 미국과 대화 가능성을 시사한 김여정 북한 노동당 부부장의 최근 담화를 관심 있게 지켜보고 있다고 7일(현 …, title: 美국무부 "北 김여정 최근 대미 담화 관심 갖고 주목" - 뉴스1, link: https://www.news1.kr/wor

In [7]:
# DuckDuckGo를 이용해 ytn.co.kr 사이트에서 로제의 신곡 APT에 대한 내용을 검색
docs = search.invoke("site:ytn.co.kr 최근 로제가 발표한 신곡은 무엇인가요?")
docs

c:\Users\pc04-06\anaconda3\envs\my_llm\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


"snippet: Nov 22, 2024 · 22일 오후 로제의 새 싱글 'number one girl' 음원과 뮤직비디오가 공개됐다. 이는 'APT.'와 마찬가지로 오는 12월 6일 발매되는 로제의 첫 번째 정규 앨범 'rosie'에도 수록될 …, title: 로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표 - 싱글리스트, link: https://www.slist.kr/news/articleView.html?idxno=598065;\nsnippet: Nov 22, 2024 · 걸그룹 블랙핑크의 멤버 로제가 22일 새 싱글 ‘넘버 원 걸 (Number one girl)’을 발표했다. 넘버 원 걸은 로제가 내달 6일에 발매하는 첫 정규앨범 ‘로지 (rosie)’의 선공개 …, title: 로제, 신곡 ‘넘버 원 걸’ 공개… 브루노 마스가 프로듀싱, link: https://biz.chosun.com/entertainment/enter_general/2024/11/22/Q6AJ6YGRUJDPNIRGX5BKHHUBBY/;\nsnippet: Nov 22, 2024 · 그룹 블랙핑크 로제가 신곡 ‘넘버 원 걸 (number one girl)’을 발표했다. 더블랙레이블은 22일 공식 SNS를 통해 로제의 새 싱글 ‘넘버 원 걸’ 음원과 뮤직비디오 공개 …, title: “팬들을 생각하며 쓴 곡”...로제, 선공개 싱글 ‘넘버 원 걸 ..., link: https://www.mk.co.kr/news/musics/11175523;\nsnippet: Dec 6, 2024 · 솔로 뮤지션으로 활약 중인 블랙핑크 로제가 아파트 신드롬에 이어 앨범 ‘rosie’를 발매했습니다., title: 오래 기다리셨습니다. 로제의 솔로 앨범 ‘rosie’가 왔어요 ..., link: https://www.fastpapermag.com/2024/12/06/오래-기다리셨습니다-로제의-솔로-앨범-rosie가-왔/"

웹페이지 내용 가져오기

In [8]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄바꿈 기준으로 분리하고, 각 결과 항목에서 링크를 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목을 출력하여 확인
    link = doc.split("link:")[1].strip()  # 각 항목에서 'link:' 이후의 URL 부분만 추출
    links.append(link)  # 추출한 링크를 리스트에 추가

# 모든 링크를 출력
print(links)

snippet: Nov 22, 2024 · 22일 오후 로제의 새 싱글 'number one girl' 음원과 뮤직비디오가 공개됐다. 이는 'APT.'와 마찬가지로 오는 12월 6일 발매되는 로제의 첫 번째 정규 앨범 'rosie'에도 수록될 …, title: 로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표 - 싱글리스트, link: https://www.slist.kr/news/articleView.html?idxno=598065
snippet: Nov 22, 2024 · 걸그룹 블랙핑크의 멤버 로제가 22일 새 싱글 ‘넘버 원 걸 (Number one girl)’을 발표했다. 넘버 원 걸은 로제가 내달 6일에 발매하는 첫 정규앨범 ‘로지 (rosie)’의 선공개 …, title: 로제, 신곡 ‘넘버 원 걸’ 공개… 브루노 마스가 프로듀싱, link: https://biz.chosun.com/entertainment/enter_general/2024/11/22/Q6AJ6YGRUJDPNIRGX5BKHHUBBY/
snippet: Nov 22, 2024 · 그룹 블랙핑크 로제가 신곡 ‘넘버 원 걸 (number one girl)’을 발표했다. 더블랙레이블은 22일 공식 SNS를 통해 로제의 새 싱글 ‘넘버 원 걸’ 음원과 뮤직비디오 공개 …, title: “팬들을 생각하며 쓴 곡”...로제, 선공개 싱글 ‘넘버 원 걸 ..., link: https://www.mk.co.kr/news/musics/11175523
snippet: Dec 6, 2024 · 솔로 뮤지션으로 활약 중인 블랙핑크 로제가 아파트 신드롬에 이어 앨범 ‘rosie’를 발매했습니다., title: 오래 기다리셨습니다. 로제의 솔로 앨범 ‘rosie’가 왔어요 ..., link: https://www.fastpapermag.com/2024/12/06/오래-기다리셨습니다-로제의-솔로-앨범-rosie가-왔/
['https://www.slist.kr/news/articleView.

In [9]:
# Langchain의 WebBaseLoader를 사용하여 웹 페이지의 내용을 불러옵니다.
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 객체를 생성. 'links'는 웹 페이지의 URL 목록을 담고 있는 변수
# bs_get_text_kwargs는 BeautifulSoup의 get_text() 메소드에 전달될 추가 인자
loader = WebBaseLoader(
    web_paths=links,  # 웹 페이지의 링크 목록을 지정
    bs_get_text_kwargs={
        "strip": True  # 웹 페이지에서 텍스트를 가져올 때 앞뒤의 공백을 제거
    },
)

# 비동기로 웹 페이지의 내용을 로드하고, 각 문서를 page_contents 리스트에 추가
page_contents = []  # 각 웹 페이지의 내용을 저장할 리스트입니다.
async for doc in loader.alazy_load():
    page_contents.append(doc)  # 불러온 문서를 page_contents 리스트에 추가

# page_contents에 있는 각 웹 페이지의 내용을 출력
for content in page_contents:
    print(content)  # 웹 페이지의 내용을 출력
    print('--------------')  # 페이지 구분을 위해 구분선을 출력

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 4/4 [00:01<00:00,  2.40it/s]


page_content='로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표 < 연예 < 문화 < 기사본문 - 싱글리스트주요서비스 바로가기본문 바로가기매체정보 바로가기로그인 바로가기기사검색 바로가기전체서비스 바로가기상단영역전체메뉴연예검색기사검색검색본문영역이전 기사보기다음 기사보기로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표바로가기복사하기본문 글씨 줄이기본문 글씨 키우기스크롤 이동 상태바현재위치홈문화연예로제, 브루노 마스 프로듀싱 신곡 'number one girl' 발표기자명정현태 기자hyntjng@slist.kr입력 2024.11.22 15:08글자크기본문 글씨 키우기본문 글씨 줄이기바로가기SNS 기사보내기페이스북(으)로 기사보내기트위터(으)로 기사보내기URL복사(으)로 기사보내기이메일(으)로 기사보내기다른 공유 찾기기사저장이 기사를 공유합니다페이스북(으)로 기사보내기트위터(으)로 기사보내기URL복사(으)로 기사보내기닫기로제가 신곡 'number one girl'을 발표했다.사진=더블랙레이블22일 오후 로제의 새 싱글 'number one girl' 음원과 뮤직비디오가 공개됐다. 이는 'APT.'와 마찬가지로 오는 12월 6일 발매되는 로제의 첫 번째 정규 앨범 'rosie'에도 수록될 예정이다. 'rosie'는 총 12곡으로 구성되어 있으며, 로제는 이번 앨범에서 전곡 작사·작곡에 참여해 여러 장르를 아우르는 자신만의 음악 세계를 보여줄 예정이다.특히 이번 신곡 'number one girl'은 로제가 팬들을 생각하며 써내려간 곡으로, 자신의 실제 경험을 녹여내기도 한 만큼 보다 진솔한 로제만의 이야기를 담고 있다. 뿐만 아니라 첫 번째 선공개 싱글 'APT.'에서 듀엣으로 호흡을 맞춘 팝스타 브루노 마스가 'number one girl' 프로듀싱에 참여하면서 더욱 풍성한 음악이 탄생했다.뮤직비디오에는 고조되는 감정을 담담히 쏟아내는 로제의 모습이 담겨 눈길을 끈다. 'number one girl'의 따뜻한 분위

In [10]:
import requests
from bs4 import BeautifulSoup

# 주어진 URL에서 기사 텍스트를 가져오는 함수
def get_article_text(url):
    try:
        # URL에 GET 요청을 보냄
        response = requests.get(url)
        # 요청이 성공하지 못하면 예외를 발생시킴
        response.raise_for_status()
        
        # BeautifulSoup을 사용하여 HTML 내용을 파싱
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 클래스가 'story-news article'인 <article> 태그를 찾음
        article = soup.find('article', class_='story-news article')
        
        # 기사를 찾았다면 그 텍스트를 반환
        if article:
            return article.get_text(strip=True)
        else:
            try:
                if soup.find('article'):
                    return soup.find('article').get_text(strip=True)
                elif soup.find('div', id="CmAdContent"):
                    return soup.find('div', id="CmAdContent").get_text(strip=True)
            except:
                return "기사 내용을 찾을 수 없습니다."
            
    # 요청이 실패할 경우 예외 처리
    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"

In [11]:
# URL 목록의 각 링크를 반복하면서 기사 텍스트를 출력
articles = []    # 가져온 내용을 리스트에 담기 위한 변수 선언
for link in links:
    print(f"URL: {link}\n")
    article_text = get_article_text(link)
    print(f"Content:\n{article_text}")
    print("--------------------------------------------------")
    articles.append(article_text)

URL: https://www.slist.kr/news/articleView.html?idxno=598065

Content:
SNS 기사보내기페이스북(으)로 기사보내기트위터(으)로 기사보내기URL복사(으)로 기사보내기이메일(으)로 기사보내기다른 공유 찾기기사저장이 기사를 공유합니다페이스북(으)로 기사보내기트위터(으)로 기사보내기URL복사(으)로 기사보내기닫기로제가 신곡 'number one girl'을 발표했다.사진=더블랙레이블22일 오후 로제의 새 싱글 'number one girl' 음원과 뮤직비디오가 공개됐다. 이는 'APT.'와 마찬가지로 오는 12월 6일 발매되는 로제의 첫 번째 정규 앨범 'rosie'에도 수록될 예정이다. 'rosie'는 총 12곡으로 구성되어 있으며, 로제는 이번 앨범에서 전곡 작사·작곡에 참여해 여러 장르를 아우르는 자신만의 음악 세계를 보여줄 예정이다.특히 이번 신곡 'number one girl'은 로제가 팬들을 생각하며 써내려간 곡으로, 자신의 실제 경험을 녹여내기도 한 만큼 보다 진솔한 로제만의 이야기를 담고 있다. 뿐만 아니라 첫 번째 선공개 싱글 'APT.'에서 듀엣으로 호흡을 맞춘 팝스타 브루노 마스가 'number one girl' 프로듀싱에 참여하면서 더욱 풍성한 음악이 탄생했다.뮤직비디오에는 고조되는 감정을 담담히 쏟아내는 로제의 모습이 담겨 눈길을 끈다. 'number one girl'의 따뜻한 분위기와 어울리는 영상미를 통해 그동안 볼 수 없었던 로제의 색다른 면면을 엿볼 수 있다.한편, 로제는 지난 10월 18일 발표한 브루노 마스와의 듀엣 곡 'APT.'를 통해 전 세계적인 신드롬을 일으키며 기록 행진을 이어가고 있다. 로제와 브루노 마스는 오늘(22일) 일본 오사카 쿄세라 돔에서 개최되는 '2024 MAMA AWARDS'에 퍼포머로 출연해 전 세계 최초로 'APT.' 무대를 선보일 예정이다.로제의 두 번째 선공개 싱글 'number one girl'이 수록된 정규 1집 'rosie'는 오는 12월 6

In [12]:
chat_history.add_message("\n".join(articles))
chat_history.add_user_message("최근 로제가 발표한 신곡은 무엇인가요?") 

# 문서 검색하고 답변을 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 
print(answer)

TypeError: sequence item 1: expected str instance, NoneType found